## 0. Dépendances

Exécute cette cellule une seule fois. Si pip modifie `transformers` ou `sentence-transformers`, redémarre ensuite le kernel.


In [1]:
%pip install -q "transformers==4.49.0" "accelerate==1.3.0" "sentence-transformers==3.4.1" rank-bm25 langgraph pandas


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Imports

In [2]:
import json
import os
import re
import numpy as np
import pandas as pd
import torch

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict

from chroma import ma_collection


c:\Users\georg\OneDrive\Bureau\ISEP\Final Project Research\projectFINAL\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Chemin utilisé : C:\Users\georg\OneDrive\Bureau\ISEP\Final Project Research\projectFINAL\chroma_3gpp_rel18
Existe : True

Contenu du dossier :
- ._chroma.sqlite3
- ._eae17378-2ffa-4d50-9800-01eaa05f2742
- chroma.sqlite3
- eae17378-2ffa-4d50-9800-01eaa05f2742

Nombre de collections : 1
[Collection(name=3gpp_rel18_collection)]
Collection(name=3gpp_rel18_collection)

Nombre de chunk : 562865
{'ids': ['2865084', '2865085', '2865086', '2865087', '2865088', '2865089', '2865090', '2865091', '2865092', '2865093'], 'embeddings': array([[-0.0092853 ,  0.01435731, -0.00187875, ...,  0.07671405,
         0.05635262, -0.00273231],
       [ 0.05003922,  0.0073945 , -0.01688451, ...,  0.00345017,
         0.04526629,  0.02224259],
       [-0.00206988,  0.02758421,  0.03543698, ..., -0.04465876,
         0.00797065,  0.01978247],
       ...,
       [-0.07843515,  0.01852478,  0.04259703, ...,  0.04828352,
         0.0727049 ,  0.02867837],
       [-0.09269179,  0.03357768,  0.0112438 , ...,  0.0197269

## 2. Charger `qa_dataset.json`

In [3]:
with open("qa_dataset.json", "r", encoding="utf-8") as f:
    raw_qa = json.load(f)

def parse_generated_qa(text):
    question = ""
    ground_truth = ""

    if "Question:" in text and "Ground truth:" in text:
        question = text.split("Question:", 1)[1].split("Ground truth:", 1)[0].strip()
        ground_truth = text.split("Ground truth:", 1)[1].strip()

    return question, ground_truth

qa_dataset = []

for item in raw_qa:
    if "question" in item and "ground_truth" in item:
        question = item["question"]
        ground_truth = item["ground_truth"]
    else:
        question, ground_truth = parse_generated_qa(item.get("generated_qa", ""))

    if question and ground_truth:
        qa_dataset.append({
            "question": question,
            "ground_truth": ground_truth
        })

print("Nombre de paires Q/R :", len(qa_dataset))
print("Première question :", qa_dataset[0]["question"])
print("Ground truth :", qa_dataset[0]["ground_truth"])


Nombre de paires Q/R : 48
Première question : ** What are the generic procedure parameters used to ensure the UE is in state RRC_connected when switching off and on the UE according to TS 38.508-1 [14] clause 4.5?

**
Ground truth : ** connectivity nr, connected without release on and test mode on


## 3. Charger le corpus ChromaDB

In [4]:
CORPUS_LIMIT = 5000

data = ma_collection.get(
    limit=CORPUS_LIMIT,
    include=["documents", "metadatas"]
)

corpus = data["documents"]
corpus_ids = data["ids"]

print("Nombre de chunks utilisés :", len(corpus))


Nombre de chunks utilisés : 5000


## 7. Retrieval hybride BM25 + BGE + RRF

## 4. BM25


In [5]:
def tokenize(text):
    return re.findall(
        r"[A-Za-z0-9]+(?:[._-][A-Za-z0-9]+)*",
        text.lower()
    )

tokenized_corpus = [tokenize(doc) for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)

print("BM25 prêt")


BM25 prêt


## 5. BGE


In [6]:
BGE_MODEL = "BAAI/bge-small-en-v1.5"
BGE_CACHE = f"bge_embeddings_{CORPUS_LIMIT}.npy"

bge_model = SentenceTransformer(BGE_MODEL)

if os.path.exists(BGE_CACHE):
    bge_embeddings = np.load(BGE_CACHE)
    print("Embeddings BGE chargés depuis le cache")
else:
    bge_embeddings = bge_model.encode(
        corpus,
        normalize_embeddings=True,
        show_progress_bar=True
    )
    np.save(BGE_CACHE, bge_embeddings)
    print("Embeddings BGE calculés et sauvegardés")

print("Shape :", bge_embeddings.shape)


Embeddings BGE chargés depuis le cache
Shape : (5000, 384)


## 6. Fusion RRF


In [7]:
def rrf_fusion(bm25_indices, bge_indices, k=60, top_k=5):
    scores = {}

    for rank, idx in enumerate(bm25_indices, start=1):
        idx = int(idx)
        scores[idx] = scores.get(idx, 0) + 1 / (k + rank)

    for rank, idx in enumerate(bge_indices, start=1):
        idx = int(idx)
        scores[idx] = scores.get(idx, 0) + 1 / (k + rank)

    ranking_final = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return ranking_final[:top_k]


In [8]:
def hybrid_retrieve(question, top_k=3):
    # BM25
    bm25_scores = bm25.get_scores(tokenize(question))
    top_bm25 = np.argsort(bm25_scores)[::-1][:10]

    # BGE
    query_embedding = bge_model.encode(
        question,
        normalize_embeddings=True
    )
    bge_scores = bge_embeddings @ query_embedding
    top_bge = np.argsort(bge_scores)[::-1][:10]

    # RRF
    fused = rrf_fusion(top_bm25, top_bge, top_k=top_k)

    indices = [idx for idx, score in fused]
    chunks = [corpus[idx] for idx in indices]
    ids = [corpus_ids[idx] for idx in indices]

    return chunks, ids


## 8. Charger `microsoft/Phi-4-mini-instruct`

In [9]:
import os
MODEL_PATH = r"C:\Users\georg\OneDrive\Bureau\ISEP\Final Project Research\projectFINAL\phi4mini-3gpp-merged-sft"

print(os.path.exists(MODEL_PATH))

print(os.listdir(MODEL_PATH))

True
['._config.json', 'added_tokens.json', 'chat_template.jinja', 'config.json', 'configuration_phi3.py', 'generation_config.json', 'merges.txt', 'model-00001-of-00002.safetensors', 'model-00002-of-00002.safetensors', 'model.safetensors.index.json', 'Modelfile', 'modeling_phi3.py', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'vocab.json']


In [10]:
import os
import json

MODEL_PATH = r"C:\Users\georg\OneDrive\Bureau\ISEP\Final Project Research\projectFINAL\phi4mini-3gpp-merged-sft"

print("Dossier existe :", os.path.isdir(MODEL_PATH))
print("\nFichiers :")

for f in os.listdir(MODEL_PATH):
    print(f)

print("\n--- tokenizer_config.json ---")
with open(os.path.join(MODEL_PATH, "tokenizer_config.json"), "r", encoding="utf-8") as f:
    tokenizer_config = json.load(f)

print(tokenizer_config)

print("\n--- config.json ---")
with open(os.path.join(MODEL_PATH, "config.json"), "r", encoding="utf-8") as f:
    config = json.load(f)

print("model_type :", config.get("model_type"))
print("architectures :", config.get("architectures"))
print("auto_map :", config.get("auto_map"))

Dossier existe : True

Fichiers :
._config.json
added_tokens.json
chat_template.jinja
config.json
configuration_phi3.py
generation_config.json
merges.txt
model-00001-of-00002.safetensors
model-00002-of-00002.safetensors
model.safetensors.index.json
Modelfile
modeling_phi3.py
special_tokens_map.json
tokenizer.json
tokenizer_config.json
vocab.json

--- tokenizer_config.json ---
{'add_bos_token': False, 'add_eos_token': False, 'add_prefix_space': False, 'added_tokens_decoder': {'199999': {'content': '<|endoftext|>', 'lstrip': False, 'normalized': False, 'rstrip': False, 'single_word': False, 'special': True}, '200018': {'content': '<|endofprompt|>', 'lstrip': False, 'normalized': False, 'rstrip': False, 'single_word': False, 'special': True}, '200019': {'content': '<|assistant|>', 'lstrip': False, 'normalized': False, 'rstrip': True, 'single_word': False, 'special': True}, '200020': {'content': '<|end|>', 'lstrip': False, 'normalized': False, 'rstrip': True, 'single_word': False, 'special

In [11]:
import os

MODEL_PATH = r"C:\Users\georg\OneDrive\Bureau\ISEP\Final Project Research\projectFINAL\phi4mini-3gpp-merged-sft"

for f in os.listdir(MODEL_PATH):
    print(repr(f))

'._config.json'
'added_tokens.json'
'chat_template.jinja'
'config.json'
'configuration_phi3.py'
'generation_config.json'
'merges.txt'
'model-00001-of-00002.safetensors'
'model-00002-of-00002.safetensors'
'model.safetensors.index.json'
'Modelfile'
'modeling_phi3.py'
'special_tokens_map.json'
'tokenizer.json'
'tokenizer_config.json'
'vocab.json'


In [12]:
# Modèle de base phi 4 MODEL_NAME = "microsoft/Phi-4-mini-instruct"

MODEL_PATH = r"C:\Users\georg\OneDrive\Bureau\ISEP\Final Project Research\projectFINAL\phi4mini-3gpp-merged-sft"
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=False
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    torch_dtype="auto",
    trust_remote_code=False
)

print("Phi-4_3gpp chargé")
print("Device :", model.device)


Loading checkpoint shards: 100%|██████████| 2/2 [00:49<00:00, 24.52s/it]
Some parameters are on the meta device because they were offloaded to the cpu and disk.


Phi-4_3gpp chargé
Device : cuda:0


## 9. Fonction de génération Phi-4

In [13]:
def generate_phi4_3gpp(user_prompt):
    messages = [
        {
            "role": "system",
            "content": "You are a technical assistant specialized in 3GPP specifications."
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return answer.strip()


## 10. LangGraph

Pipeline :

`Question → Phi-4 sans RAG → Retrieval hybride → Phi-4 avec RAG → résultat`


In [14]:
class EvaluationState(TypedDict):
    question: str
    ground_truth: str
    without_rag: str
    retrieved_context: str
    retrieved_ids: list
    with_rag: str


## 11. Nodes

In [15]:
def baseline_node(state):
    prompt = f"""
Answer the following 3GPP technical question.

Question:
{state["question"]}

Answer:
"""

    answer = generate_phi4_3gpp(prompt)

    return {"without_rag": answer}


def retrieval_node(state):
    chunks, ids = hybrid_retrieve(
        state["question"],
        top_k=3
    )

    context = "\n\n---\n\n".join(chunks)

    return {
        "retrieved_context": context,
        "retrieved_ids": ids
    }


def rag_node(state):
    prompt = f"""
Answer the technical question using only the provided 3GPP context.

If the answer is not contained in the context, say that the context does not provide enough information.

Context:
{state["retrieved_context"]}

Question:
{state["question"]}

Answer:
"""

    answer = generate_phi4_3gpp(prompt)

    return {"with_rag": answer}


## 12. Construire le graphe

In [16]:
builder = StateGraph(EvaluationState)

builder.add_node("baseline", baseline_node)
builder.add_node("retrieval", retrieval_node)
builder.add_node("rag", rag_node)

builder.add_edge(START, "baseline")
builder.add_edge("baseline", "retrieval")
builder.add_edge("retrieval", "rag")
builder.add_edge("rag", END)

evaluation_graph = builder.compile()

print("LangGraph prêt")


LangGraph prêt


## 13. Test sur une seule question

In [17]:
test_item = qa_dataset[0]

test_result = evaluation_graph.invoke({
    "question": test_item["question"],
    "ground_truth": test_item["ground_truth"],
    "without_rag": "",
    "retrieved_context": "",
    "retrieved_ids": [],
    "with_rag": ""
})

print("QUESTION")
print(test_result["question"])

print("\n1. PHI-4 SANS RAG")
print(test_result["without_rag"])

print("\n2. PHI-4 + RAG")
print(test_result["with_rag"])

print("\n3. GROUND TRUTH")
print(test_result["ground_truth"])


QUESTION
** What are the generic procedure parameters used to ensure the UE is in state RRC_connected when switching off and on the UE according to TS 38.508-1 [14] clause 4.5?

**

1. PHI-4 SANS RAG
Generic procedure parameters 1 and 2 are used to ensure the UE is in state RRC_connected when switching off and on the UE.

2. PHI-4 + RAG
Generic procedure parameters 1 and 2 are used to ensure the UE is in state RRC_connected when switching off and on the UE.

3. GROUND TRUTH
** connectivity nr, connected without release on and test mode on


## 14. Lancer les 20 questions

Cette partie fait 2 générations Phi-4 par question. Sur CPU, cela peut prendre du temps.


In [18]:
N_QUESTIONS = 20

results = []

for i, item in enumerate(qa_dataset[:N_QUESTIONS], start=1):

    result = evaluation_graph.invoke({
        "question": item["question"],
        "ground_truth": item["ground_truth"],
        "without_rag": "",
        "retrieved_context": "",
        "retrieved_ids": [],
        "with_rag": ""
    })

    results.append({
        "question": result["question"],
        "llm_without_rag": result["without_rag"],
        "llm_with_rag": result["with_rag"],
        "ground_truth": result["ground_truth"],
        "retrieved_ids": result["retrieved_ids"]
    })

    print(f"Question {i}/{N_QUESTIONS} terminée")


Question 1/20 terminée
Question 2/20 terminée
Question 3/20 terminée
Question 4/20 terminée
Question 5/20 terminée
Question 6/20 terminée
Question 7/20 terminée
Question 8/20 terminée
Question 9/20 terminée
Question 10/20 terminée
Question 11/20 terminée
Question 12/20 terminée
Question 13/20 terminée
Question 14/20 terminée
Question 15/20 terminée
Question 16/20 terminée
Question 17/20 terminée
Question 18/20 terminée
Question 19/20 terminée
Question 20/20 terminée


## 15. Tableau final

In [19]:
results_df = pd.DataFrame(results)

display(
    results_df[
        [
            "question",
            "llm_without_rag",
            "llm_with_rag",
            "ground_truth"
        ]
    ]
)


,question,llm_without_rag,llm_with_rag,ground_truth
0,** What are the generic procedure parameters u...,Generic procedure parameters 1 and 2 are used ...,Generic procedure parameters 1 and 2 are used ...,"** connectivity nr, connected without release ..."
1,If the length of the V2X service info contents...,The receiving entity shall ignore the contents...,The receiving entity shall not process the V2X...,The receiving entity shall ignore any superflu...
2,What is the name of the attribute of the media...,The attribute of the media level is called 'a=...,The attribute of the media level named 'a=mode...,"According to table A.315, the attribute name i..."
3,What is the maximum number of simultaneous MCV...,The maximum number of simultaneous MCVideo gro...,The maximum number of simultaneous MCVideo gro...,"According to table A.2-1, the maximum number o..."
4,What are the exact instants on which the value...,The values for cells at various time instants ...,The values for cells at various time instants ...,The exact instants on which these values shall...
5,What is the maximum channel bandwidth for a sy...,The maximum channel bandwidth for 64 QAM modul...,The maximum channel bandwidth for a system usi...,≤ 5.5 MHz
6,What is the maximum value for the configured U...,"23 dBm for e-UTRA bands ≤ 3GHz, except for ban...","23 dBm for e-UTRA bands ≤ 3GHz, except for ban...",6.2.5A.1_3.5-1: PCmax configured UE output pow...
7,What is the HTTP method used for the request o...,HTTP POST method is used to request the delive...,HTTP POST method is used for the request of a ...,The HTTP method used for the request of a Mess...
8,What clause specifies the details of usage of ...,Clause 5.2.2.2 specifies the details of usage ...,Clause 4.10.2.2 specifies the details of usage...,See clause 8.7.2.1.4.2.4.
9,** What is the parameter that shall be set acc...,The parameter T1 shall be set to 2 seconds.,The Operator signal threshold per access techn...,"** According to the excerpt, the parameters th..."


## 16. Sauvegarder les résultats

In [20]:
results_df.to_csv(
    "evaluation_results.csv",
    index=False,
    encoding="utf-8-sig"
)

with open("evaluation_results.json", "w", encoding="utf-8") as f:
    json.dump(
        results,
        f,
        ensure_ascii=False,
        indent=4
    )

print("Fichiers créés :")
print("- evaluation_results.csv")
print("- evaluation_results.json")


Fichiers créés :
- evaluation_results.csv
- evaluation_results.json


## Sortie finale attendue

Pour chaque question :
- **LLM completion without RAG**
- **LLM + RAG completion**
- **Ground truth completion**

Le chemin RAG est orchestré avec LangGraph et suit :

**BM25 + BGE → RRF → contexte → Phi-4**
